# 07. Characterize the Z

Fit cross sections as a function of collider energy using a scan around the Z.
This is different from plotting two-body invariant masses at one fixed beam energy.

The simple resonance model below omits photon exchange, interference, ISR,
beam spread, and acceptance variation. Use it to explore the line shape, not
to claim a precision mass or width measurement.

Copy this notebook into `work`, select **Python (hep)**, and run cells from the
top. Enter the scan points specified in your assignment.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
def four_vector(frame, prefix):
    pt, eta, phi, mass = [frame[prefix + "_" + x].to_numpy() for x in ("pt", "eta", "phi", "mass")]
    px, py, pz = pt*np.cos(phi), pt*np.sin(phi), pt*np.sinh(eta)
    return np.column_stack([np.sqrt(px*px+py*py+pz*pz+mass*mass), px, py, pz])
def pair_observables(frame):
    a, b = four_vector(frame, "l1"), four_vector(frame, "l2")
    total = a+b
    mass = np.sqrt(np.maximum(total[:,0]**2 - (total[:,1:]**2).sum(axis=1), 0))
    negative = np.where((frame.l1_charge.to_numpy() < 0)[:,None], a, b)
    momentum = np.linalg.norm(negative[:,1:], axis=1)
    cosine = np.divide(negative[:,3], momentum, out=np.full_like(momentum,np.nan), where=momentum>0)
    return mass, np.hypot(total[:,1],total[:,2]), cosine, total


In [ ]:
scan = []  # dictionaries: sqrt_s_gev, sigma_pb, error_pb; use a consistent uncertainty interpretation
if scan:
    table=pd.DataFrame(scan)
    if len(table)<5 or (table.error_pb<=0).any():
        raise ValueError("Supply at least five scan points with positive uncertainties.")
    s=table.sqrt_s_gev.to_numpy()**2
    y=table.sigma_pb.to_numpy()
    w=1/table.error_pb.to_numpy()**2
    candidates=[]
    for mass in np.linspace(88.,94.,121):
        for width in np.linspace(1.,5.,81):
            shape=s/((s-mass**2)**2+mass**2*width**2)
            amplitude=np.sum(w*shape*y)/np.sum(w*shape**2)
            chi2=np.sum(w*(y-amplitude*shape)**2)
            candidates.append((chi2,mass,width,amplitude))
    chi2,mass,width,amplitude=min(candidates)
    print("Best grid point: mass, width [GeV], diagnostic chi2 =",mass,width,chi2)
    grid=np.linspace(table.sqrt_s_gev.min(),table.sqrt_s_gev.max(),300)
    plt.errorbar(table.sqrt_s_gev,y,yerr=table.error_pb,fmt="o")
    plt.plot(grid,amplitude*grid**2/((grid**2-mass**2)**2+mass**2*width**2))
    plt.xlabel("sqrt(s) [GeV]"); plt.ylabel("Cross section [pb]"); plt.show()
# Optional asymmetry: Born two-body muons and electron beam along +z only.
angular_sample_id = None
cos_acceptance=0.8
if angular_sample_id:
    frame=get_sample(angular_sample_id).load()
    if not np.allclose(weights(frame),1):
        raise ValueError("This binomial asymmetry uncertainty assumes unit weights.")
    _,_,cosine,_=pair_observables(frame)
    forward=int(np.sum((cosine>0)&(cosine<cos_acceptance)))
    backward=int(np.sum((cosine<0)&(cosine>-cos_acceptance)))
    n=forward+backward
    if n:
        afb=(forward-backward)/n
        print("Fiducial A_FB =",afb,"+/-",np.sqrt((1-afb**2)/n))

## Questions and submission
1. Examine residuals and whether the best point hits a grid boundary; refine the grid as needed.
2. Distinguish fitting generator predictions with integration errors from fitting experimental data.
3. Identify omitted effects that could bias mass and width. Do not quote the grid step as an uncertainty.
4. Extension: measure fiducial A_FB at several energies. Specify charge, beam direction, frame and acceptance; this is not automatically the full-acceptance asymmetry.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.